# Daily forecast of financial time series group questions

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%matplotlib inline
import matplotlib.pylab as plt

## Get all questions

In [3]:
from get_market_pulse_25q3_questions import get_market_pulse_25q3_questions
ifps = get_market_pulse_25q3_questions()

200


In [4]:
import json, os
if os.path.exists('results.json'):
    with open('results.json', 'r') as f:
        results = json.load(f)
else:
    from get_underlying_urls import get_underlying_urls
    results = get_underlying_urls(ifps)
    with open('results.json', 'w') as f:
        json.dump(results, f, indent=4)

## Acquire question data

In [5]:
from urls_to_ticker import urls_to_ticker
from pprint import pprint
sources = []
for key, (title, urls) in results.items():
    sources.extend(urls_to_ticker(urls))

In [6]:
sources = list(sorted(set(sources)))

In [7]:
import os, joblib
from get_data_for import get_data_for
if os.path.exists('history.joblib'):
    history = joblib.load('history.joblib')
else:
    history = {(x,y,z): get_data_for(x,y,z) for x,y,z in sources}
    joblib.dump(history, 'history.joblib');

In [8]:
group_id_to_data = {int(x): results[x] for x in results}

In [9]:
results = {int(key): value for key, value in results.items()}

In [10]:
for ifp in ifps:
    ifp['sources'] = urls_to_ticker(results[ifp['group']['id']][1])
    ifp['data'] = {x: history[x] for x in ifp['sources']}
    if 'following companies' in ifp['title']:
        company = ifp['title'].split('(')[1].split(')')[0]
        ifp['sources'] = [(x,y,z) for x,y,z in ifp['sources'] if y == company]
        ifp['data'] = {(x,y,z):w for (x,y,z),w in ifp['data'].items() if y == company}

## Get forward period start and end date

In [11]:
from get_forward_period_start_and_end_date import get_forward_period_start_and_end_date

In [13]:
for ifp in ifps:
    if 'period' not in ifp:
        ifp['period'] = get_forward_period_start_and_end_date(ifp)

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.265477188428243

```python
["2025-07-21", "2025-08-01"]
```
START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.016621951262156168

```python
["2025-08-04", "2025-08-15"]
```
START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.017175745964050294

```python
["2025-07-21", "2025-08-01"]
```
START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.017060001691182453

```python
["2025-08-04", "2025-08-15"]
```
START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.016288824876149497

```python
["2025-07-21", "2025-08-01"]
```
START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.0178508798281351

## Parse observable of data

In [14]:
from get_observable import get_observable
import pandas as pd
pd.set_option('display.max_colwidth', 1000)

In [15]:
from get_period_type import get_period_type

In [16]:
for i, ifp in enumerate(ifps):
    if 'observable' not in ifp:
        ifp['observable'] = get_observable(ifp)
        ifp['period_type'] = get_period_type(ifp)

In [17]:
with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

## Generate forecast for each question

In [ ]:
from forecast_high_on_any_day_in_forward_period import forecast_high_on_any_day_in_forward_period
from forecast_close_on_last_day_of_period import forecast_close_on_last_day_of_period
from forecast_stock_price_return_spread import forecast_stock_price_return_spread
from forecast_futures_total_price_return_spread import forecast_futures_total_price_return_spread
from forecast_quarter_diluted_eps_on_next_filing_date import forecast_quarter_diluted_eps_on_next_filing_date

for i, ifp in enumerate(ifps):
    print(f"[{i}] {ifp['title']}")
    ### High on any day in forward biweekly period from today
    if ifp['observable'] == 'High' and ifp['period_type'] == 'any day in biweekly period':
        rng, prediction = forecast_high_on_any_day_in_forward_period(ifp)
    ### Close on last day of biweekly period
    elif ifp['observable'] == 'Close' and ifp['period_type'] == 'last day of biweekly period':
        rng, prediction = forecast_close_on_last_day_of_period(ifp)
    ### period stock price return on last vs first day of biweekly period
    elif (ifp['observable'], ifp['period_type']) == ('stock price Close return', 'return on last vs first day of biweekly period'):
        rng, prediction = forecast_stock_price_return_spread(ifp)
    ### period futures total price return on last vs first day of biweekly period
    elif (ifp['observable'], ifp['period_type']) == ('futures total price Close return', 'return on last vs first day of biweekly period'):
        rng, prediction = forecast_futures_total_price_return_spread(ifp)
    ### quarter diluted eps on next SEC filing date	
    elif (ifp['observable'], ifp['period_type']) == ('quarter diluted eps', 'next SEC filing date'):
        rng, prediction = forecast_quarter_diluted_eps_on_next_filing_date(ifp)
    ### quarter total revenue on next SEC filing date	
    elif ('quarter total revenue', 'next SEC filing date') == ('quarter total revenue', 'next SEC filing date'):
        rng, prediction = forecast_quarter_diluted_eps_on_next_filing_date(ifp)
    else:
        raise Exception(f"Unhandled question [{ifp['id']}] {ifp['title']}")
    ifp['prediction'] = prediction

## Submit the question

In [126]:
import pandas as pd
import numpy as np
from ensure_min_increase import ensure_min_increase
from post_group_forecast import post_group_forecast

for ifp in ifps[1:]:
    break

In [127]:
row = pd.Series()
row['id_of_question'] = ifp['id']
row['id_of_post'] = ifp['post_id']
row['question_type'] = 'numeric'
row['forecast'] = """Statistical analysis."""
p1 = ifp['prediction']

In [174]:
p1[0] = ifp['scaling']['range_min']
p1[-1] = ifp['scaling']['range_max']

In [175]:
p2 = dict(zip(rng,p1))

In [176]:
p2

{0.0: 10.0,
 0.5: 11.239999771118164,
 1.0: 11.239999771118164,
 1.5: 11.91234983921051,
 2.0: 13.000199832916259,
 2.5: 13.739999771118164,
 3.0: 13.779899778366088,
 3.5: 13.95439980506897,
 4.0: 14.023599739074706,
 4.5: 14.029999732971191,
 5.0: 14.166499757766724,
 5.5: 14.250300006866455,
 6.0: 14.271400127410889,
 6.5: 14.297349896430969,
 7.0: 14.322199745178223,
 7.5: 14.339500141143798,
 8.0: 14.382000312805175,
 8.5: 14.411150155067444,
 9.0: 14.44280005455017,
 9.5: 14.460000038146973,
 10.0: 14.52899990081787,
 10.5: 14.691649618148803,
 11.0: 14.699999809265137,
 11.5: 14.699999809265137,
 12.0: 14.699999809265137,
 12.5: 14.712500095367432,
 13.0: 14.862100248336793,
 13.5: 15.010000228881836,
 14.0: 15.093600254058838,
 14.5: 15.410400323867798,
 15.0: 15.61800012588501,
 15.5: 15.646299710273743,
 16.0: 15.677199592590332,
 16.5: 15.6899995803833,
 17.0: 15.6899995803833,
 17.5: 15.810999464988708,
 18.0: 16.166399192810058,
 18.5: 16.390049390792846,
 19.0: 16.3986995

In [185]:
from metaculus_generate_continuous_cdf import metaculus_generate_continuous_cdf

In [186]:
p4 = metaculus_generate_continuous_cdf(
                    p2,
                    ifp)

NameError: name 'nominal_location_to_cdf_location' is not defined

In [179]:
from standardize_cdf import standardize_cdf

In [180]:
p5 = standardize_cdf(p4, ifp['scaling'])

In [183]:
row['prediction'] = p5

In [184]:
post_group_forecast(row)

Prediction Post status code: 201
Posted forecast for 38212


## Assess crowd alignment